# Worked solutions: 07 TB chest X-ray screening

**Run top to bottom: about 2-3 minutes, CPU only. It changes nothing of yours.** It reads `data/` and the participant table, and writes no files, no MLflow runs and no registered models.

The next cell repeats the lesson's setup (same data, same `STEP1`..`STEP4`, same model, same threshold), so the numbers match `tb_screening.ipynb`. Each exercise below has the question, a few short code cells, and an **Answer** based on the numbers printed above it.

Groups here are small (7 TB cases at Chainda South, 24 previously treated TB cases at Kanyama, 26 per half in Exercise 4). Read the intervals and the repeated-split ranges, not a single number.

In [1]:
# Setup: the same lines as tb_screening.ipynb (data, STEP1..STEP4, model, threshold, checks).
# Nothing here writes anything: no MLflow, no files.
import json
import re
import time
import warnings

warnings.filterwarnings("ignore")
import nbformat
import numpy as np
import pandas as pd
from scipy.stats import beta
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_predict, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import tb_helpers as tb

T0 = time.time()
pd.set_option("display.width", 200, "display.max_columns", 30, "display.precision", 3)
people, X = tb.load_data()       # one row per participant (+ their X-ray's header facts); X = 1024 image features each
y = people.tb.to_numpy()         # 1 = bacteriologically confirmed TB (culture or Xpert)


def group(clinic, hiv):
    """Row positions of one clinic's HIV-negative or HIV-positive participants."""
    return np.flatnonzero((people.clinic == clinic).to_numpy() & (people.hiv == hiv).to_numpy())


STEP1 = group("Kanyama", "HIV-negative")
STEP2 = group("Chawama", "HIV-negative")
STEP3 = group("Chainda South", "HIV-negative")
STEP4 = group("Kanyama", "HIV-positive")

# Step 1, exactly as in the lesson: 70% of Kanyama HIV-negative to train, threshold for 90% sensitivity from CV
train, test = train_test_split(STEP1, test_size=0.3, stratify=y[STEP1], random_state=0)


def new_model():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.01, class_weight="balanced", max_iter=5000))


p_cv = cross_val_predict(new_model(), X[train], y[train], cv=5, method="predict_proba")[:, 1]
threshold = tb.threshold_for_sensitivity(y[train], p_cv, 0.90)
model = new_model().fit(X[train], y[train])
score = lambda idx: model.predict_proba(X[idx])[:, 1]
google = lambda idx: people.google_score.to_numpy()[idx]
reference = people.iloc[train]


def evaluate(sets, scorer=score, thr=None):
    thr = threshold if thr is None else thr
    return pd.DataFrame({name: tb.screening_metrics(y[idx], scorer(idx), thr) for name, idx in sets.items()}).T


# the lesson's label-free checks (same definitions as tb_screening.ipynb)
def header_check(idx, min_share=0.05):
    batch = people.iloc[idx]
    out = {}
    for c in ["Manufacturer", "PhotometricInterpretation", "PixelSpacing", "Columns"]:
        usual = reference[c].value_counts(normalize=True)
        out[c] = (~batch[c].isin(usual[usual >= min_share].index)).mean()
    return pd.Series(out, name="share not matching reference")


def feature_drift(idx):
    both = np.r_[train, idx]
    which = np.r_[np.zeros(len(train)), np.ones(len(idx))]
    return cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)),
                           X[both], which, cv=5, scoring="roc_auc").mean()


# sanity check against the lesson: step 1 on the 155 held-out people
print(f"threshold {threshold:.3f}")
evaluate({"1. Kanyama, HIV-negative (held out)": test})

threshold 0.115


,people,TB cases,roc_auc,sensitivity,specificity,flag_rate
"1. Kanyama, HIV-negative (held out)",155.0,25.0,0.804,0.84,0.515,0.542


## Exercise 1

**Previous TB.** Repeat step 4's comparison for people previously treated for TB (`prev_tb`), with Google's model. (Old TB scars look like TB on an X-ray: expect a big change in specificity.)

In [2]:
# Step 4 compared two groups at the same clinic. Here: Kanyama, people never treated for TB vs previously treated.
# Google's model, at the study's two thresholds (0.305 = the lesson's; 0.465 = the study's stricter, "exploratory" one).
KAN = np.flatnonzero((people.clinic == "Kanyama").to_numpy())
prev = people.prev_tb.to_numpy()                    # "yes" / "no" / a few missing
groups = {"never treated (no)": KAN[prev[KAN] == "no"], "previously treated (yes)": KAN[prev[KAN] == "yes"]}
rows = []
for thr in (0.305, 0.465):
    for name, idx in groups.items():
        m = tb.screening_metrics(y[idx], google(idx), thr)
        rows.append({"threshold": thr, "prev_tb": name, **{k: m[k] for k in ("people", "TB cases", "roc_auc", "sensitivity", "specificity", "flag_rate")}})
ex1 = pd.DataFrame(rows).set_index(["threshold", "prev_tb"])
ex1

people  TB cases  roc_auc  sensitivity  specificity  flag_rate
threshold prev_tb                                                                                 
0.305     never treated (no)           690       110    0.903        0.891        0.672      0.417
          previously treated (yes)     171        24    0.712        0.875        0.279      0.743
0.465     never treated (no)           690       110    0.903        0.809        0.826      0.275
          previously treated (yes)     171        24    0.712        0.792        0.442      0.591

In [3]:
# How sure are we? 90% bootstrap intervals for specificity (non-TB people only), and the change between the groups.
def boot_spec(idx, thr, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    neg = idx[y[idx] == 0]
    return np.array([(google(rng.choice(neg, len(neg))) < thr).mean() for _ in range(n)])


rows = []
for thr in (0.305, 0.465):
    a, b = (boot_spec(groups[k], thr) for k in groups)
    rows.append({"threshold": thr,
                 "specificity, never treated": f"{a.mean():.2f} ({np.percentile(a, 5):.2f}-{np.percentile(a, 95):.2f})",
                 "specificity, previously treated": f"{b.mean():.2f} ({np.percentile(b, 5):.2f}-{np.percentile(b, 95):.2f})",
                 "change": f"{(b - a).mean():+.2f} ({np.percentile(b - a, 5):+.2f} to {np.percentile(b - a, 95):+.2f})"})
pd.DataFrame(rows).set_index("threshold")

,"specificity, never treated","specificity, previously treated",change
threshold,,,
0.305,0.67 (0.64-0.70),0.28 (0.22-0.33),-0.39 (-0.46 to -0.32)
0.465,0.83 (0.80-0.85),0.44 (0.37-0.50),-0.39 (-0.45 to -0.32)


In [4]:
# Is it just Kanyama? Same comparison on all three clinics (Google's model copes with the Konica machine).
rows = []
for thr in (0.305, 0.465):
    for name, val in [("never treated (no)", "no"), ("previously treated (yes)", "yes")]:
        idx = np.flatnonzero(prev == val)
        m = tb.screening_metrics(y[idx], google(idx), thr)
        rows.append({"threshold": thr, "prev_tb": name, **{k: m[k] for k in ("people", "TB cases", "sensitivity", "specificity", "flag_rate")}})
pd.DataFrame(rows).set_index(["threshold", "prev_tb"])

people  TB cases  sensitivity  specificity  flag_rate
threshold prev_tb                                                                        
0.305     never treated (no)          1522       169        0.846        0.763      0.304
          previously treated (yes)     302        36        0.889        0.361      0.669
0.465     never treated (no)          1522       169        0.746        0.882      0.187
          previously treated (yes)     302        36        0.833        0.538      0.507

**Answer.** Specificity drops by about 0.39 for people previously treated for TB, at both thresholds. Sensitivity does not move.

- Kanyama, threshold 0.305: specificity 0.67 (0.64-0.70) for never treated vs 0.28 (0.22-0.33) for previously treated; change -0.39 (-0.46 to -0.32). The flag rate goes from 42% to 74%.
- Threshold 0.465 (the study's stricter one; the lesson notebook only uses 0.305): 0.83 vs 0.44, the same -0.39. The flag rate goes from 28% to 59%. A stricter threshold raises specificity for both groups but does not close the gap.
- Sensitivity is about the same (0.89 vs 0.88 at 0.305), but there are only 24 TB cases among the previously treated, so it is uncertain by roughly +/-0.15. ROC-AUC falls from 0.90 to 0.71.
- All three clinics together (302 previously treated, 36 TB cases): specificity 0.76 vs 0.36 at 0.305, and 0.88 vs 0.54 at 0.465. Same story.

Why: old scars look like active TB, so the model sends most previously treated people for a lab test who do not need one. Unlike HIV status in step 4, this one is **specificity**, not sensitivity. Caveat: previous TB and HIV overlap (100 of 346 HIV-positive Kanyama participants were previously treated, vs 71 of 515 HIV-negative), and this comparison does not adjust for that.

## Exercise 2

**A header rule that blocks.** Write a rule that would stop Chainda South's batch before scoring. Should it block or warn, and who gets the message?

In [5]:
# The rule. It only looks at header columns, so it can run before any image is scored.
HEADER_FIELDS = ["Manufacturer", "PhotometricInterpretation", "PixelSpacing", "Columns"]   # as in the lesson's header check
# what is "normal": values that made up at least 5% of the reference (step 1's training images)
NORMAL = {c: set(vc[vc >= 0.05].index) for c in HEADER_FIELDS for vc in [reference[c].value_counts(normalize=True)]}


def header_rule(batch_idx, block_at=0.50, warn_at=0.05):
    """Decide before scoring. Returns a dict:
    action    "ok" | "warn" | "block"
    notify    who should get the message
    message   technical text for the pipeline owner
    message_for_site   plain text for the clinic
    mismatch  share of the batch with an unusual value, per header field

    block: at least half the batch is unlike anything the model was built on -> do not score it.
    warn : a noticeable minority is unlike -> score, but tell someone to look.
    """
    batch = people.iloc[batch_idx]
    share = {c: float((~batch[c].isin(NORMAL[c])).mean()) for c in HEADER_FIELDS}
    top = max(share.values())
    odd = [c for c in HEADER_FIELDS if share[c] >= (block_at if top >= block_at else warn_at)]
    seen = ", ".join(f"{c} = {batch[c].value_counts().index[0]}" for c in odd)
    if top >= block_at:
        return {"action": "block", "notify": ["pipeline owner (on-call)", "programme manager / site lead"],
                "message": f"BLOCKED, not scored: {top:.0%} of {len(batch)} images differ from the training data ({seen}).",
                "message_for_site": "The AI result is switched off for this batch. Keep using the usual pathway; we are checking and will tell you.",
                "mismatch": share}
    if top >= warn_at:
        return {"action": "warn", "notify": ["pipeline owner (on-call)"],
                "message": f"WARNING, scored: {top:.0%} of {len(batch)} images are unusual ({seen}).",
                "message_for_site": "", "mismatch": share}
    return {"action": "ok", "notify": [], "message": f"ok: headers match the reference ({len(batch)} images)",
            "message_for_site": "", "mismatch": share}




In [6]:
batches = {"1. Kanyama HIV-neg (held out)": test, "2. Chawama": STEP2, "3. Chainda South": STEP3, "4. Kanyama HIV-pos": STEP4}
for name, idx in batches.items():
    r = header_rule(idx)
    print(f"{name:30s} {r['action']:6s} {r['message']}")

1. Kanyama HIV-neg (held out)  ok     ok: headers match the reference (155 images)
2. Chawama                     ok     ok: headers match the reference (512 images)
3. Chainda South               block  BLOCKED, not scored: 99% of 148 images differ from the training data (Manufacturer = KONICA MINOLTA, PhotometricInterpretation = MONOCHROME2, PixelSpacing = [0.175, 0.175], Columns = 2010).
4. Kanyama HIV-pos             ok     ok: headers match the reference (346 images)


In [7]:
# Two more checks: the pipeline-bug batch (same headers as step 3, so same decision), and a mixed batch
# (Kanyama + 20% Chainda South images: should warn, not block).
rng = np.random.default_rng(0)
mixed = np.r_[rng.choice(STEP1, 100, replace=False), rng.choice(STEP3, 25, replace=False)]
konica = [i for i in STEP3 if people.PhotometricInterpretation.iloc[i] == "MONOCHROME2"]      # the lesson's bug batch
for name, idx in {"3b. Chainda South, bug batch": konica, "mixed: 100 Kanyama + 25 Chainda South": mixed}.items():
    r = header_rule(idx)
    print(f"{name:40s} {r['action']:6s} {r['message']}")
print()
print(pd.DataFrame({n: header_rule(i)["mismatch"] for n, i in batches.items()}).T.round(3))

3b. Chainda South, bug batch             block  BLOCKED, not scored: 100% of 147 images differ from the training data (Manufacturer = KONICA MINOLTA, PhotometricInterpretation = MONOCHROME2, PixelSpacing = [0.175, 0.175], Columns = 2010).
mixed: 100 Kanyama + 25 Chainda South    warn   WARNING, scored: 22% of 125 images are unusual (Manufacturer = FUJIFILM Corporation, PhotometricInterpretation = MONOCHROME1, PixelSpacing = [0.15, 0.15], Columns = 2836).

                               Manufacturer  PhotometricInterpretation  PixelSpacing  Columns
1. Kanyama HIV-neg (held out)         0.013                      0.013         0.013    0.026
2. Chawama                            0.000                      0.000         0.000    0.000
3. Chainda South                      0.993                      0.993         0.993    0.993
4. Kanyama HIV-pos                    0.014                      0.014         0.014    0.026


**Answer.** `header_rule` looks only at header columns, so it runs before any image is scored. It blocks Chainda South (99% of 148 images differ in manufacturer, photometric interpretation, pixel spacing and columns), and the pipeline-bug batch gets the same decision because its headers are the same. Steps 1, 2 and 4 pass; a mixed batch (22% Chainda images) only warns.

**Block or warn?** Block when most of the batch is unlike the training data. At Chainda South the model gave valid probabilities and found 0 of 7 TB cases, with no error. A warning read after scoring comes too late: people have already been told "negative". Warn when a minority is odd (Kanyama has a second machine at about 3%; the rule is quiet below 5%). The 50% and 5% cut-offs are choices: tune them on past batches.

**Who gets the message?** Not only the person looking at the results screen.
- The **pipeline owner / on call** gets the technical message (which fields, how many images): they can fix the pipeline.
- The **programme manager or site lead** gets the decision: AI result off for this clinic, what to do meanwhile, whether to collect local labelled images.
- The **clinic** gets one plain sentence: switched off for this batch, keep the usual pathway. A block must never mean "no screening". 
- Anyone allowed to override should be named, and the override logged.

## Exercise 3

**Train on both machines.** Retrain with Kanyama plus half of Chainda South's images. What happens at each site? Why is 11 TB cases not enough?

In [8]:
# Chainda South: 224 people (HIV-negative and positive), of whom only 11 have TB.
CHAI = np.flatnonzero((people.clinic == "Chainda South").to_numpy())
print(f"Chainda South: {len(CHAI)} people, {y[CHAI].sum()} with TB")


def retrain(seed):
    """Kanyama (step 1's training set) + a random half of Chainda South; test on the other half and on Kanyama held-out."""
    chai_tr, chai_te = train_test_split(CHAI, test_size=0.5, stratify=y[CHAI], random_state=seed)
    idx = np.r_[train, chai_tr]
    p = cross_val_predict(new_model(), X[idx], y[idx], cv=5, method="predict_proba")[:, 1]
    thr = tb.threshold_for_sensitivity(y[idx], p, 0.90)             # same recipe as the lesson
    m2 = new_model().fit(X[idx], y[idx])
    is_chai = np.r_[np.zeros(len(train), bool), np.ones(len(chai_tr), bool)]
    cv_tb = {"Kanyama": np.median(p[~is_chai & (y[idx] == 1)]), "Chainda South": np.median(p[is_chai & (y[idx] == 1)])}
    out = {"threshold: pooled model (lesson: %.3f)" % threshold: thr, "TB cases in Chainda training half": int(y[chai_tr].sum()), "TB cases in Chainda test half": int(y[chai_te].sum()),
           "median cross-validated score of TB cases, Kanyama": cv_tb["Kanyama"], "median cross-validated score of TB cases, Chainda South": cv_tb["Chainda South"]}
    for site, te in [("Kanyama", test), ("Chainda South", chai_te)]:
        old = tb.screening_metrics(y[te], score(te), threshold)
        new = tb.screening_metrics(y[te], m2.predict_proba(X[te])[:, 1], thr)
        for k in ("roc_auc", "sensitivity", "specificity"):
            out[f"{site} {k}: lesson model"] = old[k]
            out[f"{site} {k}: retrained"] = new[k]
    return out


one = retrain(0)
pd.Series(one).to_frame("seed 0")

Chainda South: 224 people, 11 with TB


,seed 0
threshold: pooled model (lesson: 0.115),0.026
TB cases in Chainda training half,6.000
TB cases in Chainda test half,5.000
"median cross-validated score of TB cases, Kanyama",0.594
"median cross-validated score of TB cases, Chainda South",0.013
Kanyama roc_auc: lesson model,0.804
Kanyama roc_auc: retrained,0.797
Kanyama sensitivity: lesson model,0.840
Kanyama sensitivity: retrained,1.000
Kanyama specificity: lesson model,0.515


In [9]:
# With 5-6 TB cases in each half, one split is mostly luck. Repeat with 20 different random halves.
reps = pd.DataFrame([retrain(s) for s in range(20)])
metrics = [("Kanyama", "roc_auc"), ("Kanyama", "specificity"), ("Kanyama", "sensitivity"), ("Chainda South", "roc_auc"), ("Chainda South", "sensitivity"), ("Chainda South", "specificity")]
fmt = lambda col: "{:.2f}  ({:.2f}-{:.2f})".format(reps[col].mean(), reps[col].min(), reps[col].max())
summary = pd.DataFrame({f"{site}: {k}": {"lesson model": fmt(f"{site} {k}: lesson model"), "retrained": fmt(f"{site} {k}: retrained")}
                        for site, k in metrics}).T.rename_axis("mean (min-max) over 20 splits")
print("TB cases in the Chainda training half:", sorted(int(v) for v in reps["TB cases in Chainda training half"].unique()),
      "| in the test half:", sorted(int(v) for v in reps["TB cases in Chainda test half"].unique()))
summary

TB cases in the Chainda training half: [5, 6] | in the test half: [5, 6]


,lesson model,retrained
mean (min-max) over 20 splits,,
Kanyama: roc_auc,0.80 (0.80-0.80),0.79 (0.78-0.80)
Kanyama: specificity,0.52 (0.52-0.52),0.06 (0.02-0.12)
Kanyama: sensitivity,0.84 (0.84-0.84),1.00 (1.00-1.00)
Chainda South: roc_auc,0.67 (0.57-0.82),0.60 (0.42-0.79)
Chainda South: sensitivity,0.00 (0.00-0.00),0.89 (0.80-1.00)
Chainda South: specificity,1.00 (0.99-1.00),0.27 (0.06-0.51)


In [10]:
# How much can 5-6 cases tell us? 90% confidence interval (Clopper-Pearson) for a sensitivity measured on n cases.
def cp(k, n, a=0.10):
    return (beta.ppf(a / 2, k, n - k + 1) if k else 0.0, beta.ppf(1 - a / 2, k + 1, n - k) if k < n else 1.0)


pd.DataFrame([{"TB cases found / tested": f"{k} / {n}", "sensitivity": k / n, "90% interval": "{:.2f} - {:.2f}".format(*cp(k, n))}
              for k, n in [(5, 5), (4, 5), (3, 5), (5, 6), (4, 6), (3, 6)]]).set_index("TB cases found / tested")

,sensitivity,90% interval
TB cases found / tested,,
5 / 5,1.000,0.55 - 1.00
4 / 5,0.800,0.34 - 0.99
3 / 5,0.600,0.19 - 0.92
5 / 6,0.833,0.42 - 0.99
4 / 6,0.667,0.27 - 0.94
3 / 6,0.500,0.15 - 0.85


**Answer.** The retrained model is not better at Chainda South, and it is worse at Kanyama. Chainda South has 11 TB cases in total, so the training half holds only 5 or 6, and the test half 5 or 6.

- **Chainda South (test half):** ROC-AUC 0.67 (0.57-0.82) for the lesson model vs 0.60 (0.42-0.79) retrained, across 20 random halves. The ranking did not improve; the ranges overlap completely. Sensitivity rises from 0 to 0.89, but only because the new threshold flags most people: specificity 0.27.
- **Kanyama (held out):** ROC-AUC is about the same (0.79 vs 0.80), but specificity falls from 0.52 to 0.06: almost everybody is flagged. Reason: the threshold is set for 90% sensitivity over *all* TB cases, and the 5-6 Chainda cases still score near zero in cross-validation (median 0.013, vs 0.59 for Kanyama's cases). So the threshold is dragged down to 0.026 (lesson: 0.115), which hurts the site where the model did work.
- **Why 11 is not enough:** 5-6 positives cannot teach 1024 features what TB looks like on a new machine; a threshold set on them is set by a handful of people; and a test on 5-6 cases cannot tell good from bad (finding 5 of 5 still gives an interval of 0.55-1.00; 4 of 6 gives 0.27-0.94). Even the splits vary (AUC 0.42 to 0.79 for the same recipe).

What would help: many more labelled local cases (hundreds), per-site thresholds, standardised acquisition, or a model already trained on many machines (Google's, in the lesson).

## Exercise 4

**Group thresholds.** Choose a separate threshold for HIV-positive patients (on half of them) that restores 90% sensitivity. What does it cost in false alarms? Is a group-specific threshold acceptable here?

In [11]:
# Step 4's group: Kanyama HIV-positive (346 people, 52 TB). Split in two halves, keeping TB cases balanced.
# Choose the threshold on half A for 90% sensitivity; judge it on half B (never used for choosing).
def group_threshold(scorer, seed, target=0.90):
    A, B = train_test_split(STEP4, test_size=0.5, stratify=y[STEP4], random_state=seed)
    thr = tb.threshold_for_sensitivity(y[A], scorer(A), target)
    return A, B, thr


def compare(scorer, thr_old, seed):
    A, B, thr_new = group_threshold(scorer, seed)
    old = tb.screening_metrics(y[B], scorer(B), thr_old)
    new = tb.screening_metrics(y[B], scorer(B), thr_new)
    return {"half B people": old["people"], "half B TB": old["TB cases"], "new threshold": thr_new,
            "sens, old": old["sensitivity"], "sens, new": new["sensitivity"],
            "spec, old": old["specificity"], "spec, new": new["specificity"]}


models = {"our model (lesson threshold)": (score, threshold), "Google's model (0.305)": (google, 0.305)}
one = pd.DataFrame({name: compare(sc, thr, seed=0) for name, (sc, thr) in models.items()}).T
one["extra false alarms per 100 without TB (negative = fewer)"] = (one["spec, old"] - one["spec, new"]) * 100
one

,half B people,half B TB,new threshold,"sens, old","sens, new","spec, old","spec, new",extra false alarms per 100 without TB (negative = fewer)
our model (lesson threshold),173.0,26.0,0.153,0.808,0.769,0.449,0.571,-12.245
Google's model (0.305),173.0,26.0,0.625,0.692,0.423,0.571,0.864,-29.252


In [12]:
# One split is a coin toss with 26 TB cases per half. Repeat 50 times (different random halves).
rows, rep4 = {}, {}
for name, (sc, thr) in models.items():
    r = rep4[name] = pd.DataFrame([compare(sc, thr, s) for s in range(50)])
    r["extra FA"] = (r["spec, old"] - r["spec, new"]) * 100
    rows[name] = {"threshold: old": thr, "threshold: new (mean, range)": f"{r['new threshold'].mean():.3f} ({r['new threshold'].min():.3f}-{r['new threshold'].max():.3f})",
                  "sensitivity on half B, old": r["sens, old"].mean(),
                  "sensitivity on half B, new": r["sens, new"].mean(),
                  "  ... range": f"{r['sens, new'].min():.2f}-{r['sens, new'].max():.2f}",
                  "specificity, old": r["spec, old"].mean(), "specificity, new": r["spec, new"].mean(),
                  "extra false alarms per 100 without TB (mean)": r["extra FA"].mean()}
pd.DataFrame(rows).T.round(3).T

,our model (lesson threshold),Google's model (0.305)
threshold: old,0.115,0.305
"threshold: new (mean, range)",0.088 (0.033-0.198),0.242 (0.194-0.625)
"sensitivity on half B, old",0.848,0.804
"sensitivity on half B, new",0.873,0.862
... range,0.65-1.00,0.42-1.00
"specificity, old",0.454,0.572
"specificity, new",0.387,0.397
extra false alarms per 100 without TB (mean),6.68,17.456


In [13]:
# The cost in people: scale the 50-split averages to 1,000 HIV-positive attendees with Step 4's TB prevalence (52 of 346).
prev4 = y[STEP4].mean()
rows = []
for name, r in rep4.items():
    for label, sens, spec in [("one threshold for everyone", r["sens, old"].mean(), r["spec, old"].mean()),
                              ("separate HIV-positive threshold", r["sens, new"].mean(), r["spec, new"].mean())]:
        rows.append({"model": name.split(" (")[0], "threshold": label,
                     "TB cases found (of %d)" % round(1000 * prev4): round(1000 * prev4 * sens),
                     "false alarms (of %d without TB)" % round(1000 * (1 - prev4)): round(1000 * (1 - prev4) * (1 - spec)),
                     "sent for lab test": round(1000 * (prev4 * sens + (1 - prev4) * (1 - spec)))})
pd.DataFrame(rows).set_index(["model", "threshold"])

TB cases found (of 150)  false alarms (of 850 without TB)  sent for lab test
model          threshold                                                                                                    
our model      one threshold for everyone                           127                               464                591
               separate HIV-positive threshold                      131                               521                652
Google's model one threshold for everyone                           121                               364                485
               separate HIV-positive threshold                      130                               512                642

**Answer.** We split Kanyama's 346 HIV-positive participants in halves (26 TB cases each), chose the threshold for 90% sensitivity on half A and judged it on half B.

- **Which model needs it:** Google's model: sensitivity 0.94 for HIV-negative vs 0.81 for HIV-positive at 0.305 (the lesson's numbers). Our model at its lesson threshold is already at 0.85 for HIV-positive, so there is little to restore (the table shows both).
- **Does it restore 90%?** Not reliably. Over 50 random splits, Google's model at the group threshold (about 0.24) reaches sensitivity 0.86 on average on the other half, with a range of 0.42 to 1.00. 26 cases are too few to pin down a 90% cut-off. The first split shown (seed 0) is an unlucky one: use the 50-split averages.
- **Cost in false alarms:** specificity falls from 0.57 to 0.40, i.e. about 17 more false alarms per 100 people without TB. Per 1,000 HIV-positive attendees (about 150 with TB): 121 to 130 TB cases found (+9), 364 to 512 false alarms (+148): about 16 extra lab tests for each extra case found.

**Is it acceptable?** Technically yes, and it is the same logic as the 90% rule: HIV-positive patients have more TB and worse outcomes if it is missed, and the cost is a lab test. But only with conditions: the threshold needs far more local lab results than 26 cases per half (or show the interval and re-check as data arrives); HIV status is sensitive and must be used for this and nothing else; the extra workload on the lab and the clinic is planned for; and it is a patch. Fixing the model with HIV-positive training data is the better answer. This is a clinical and ethics decision for the programme, not only a data one.

## Exercise 5

**Shortcut learning.** Train a model on TB cases from Kanyama and non-TB people from Chainda South only. How well does it do on its own mix, and on Kanyama alone? What did it learn?

In [14]:
# Build the confounded training set. Every TB case comes from Kanyama, every non-TB person from Chainda South.
KAN = np.flatnonzero((people.clinic == "Kanyama").to_numpy())
CHAI0 = CHAI[y[CHAI] == 0]                                    # Chainda South, no TB (213)
kan_tr, kan_te = train_test_split(KAN, test_size=0.3, stratify=y[KAN], random_state=0)
chai_tr, chai_te = train_test_split(CHAI0, test_size=0.3, random_state=0)
tr_idx = np.r_[kan_tr[y[kan_tr] == 1], chai_tr]               # Kanyama TB + Chainda non-TB
print(f"training set: {y[tr_idx].sum()} TB cases (all Kanyama) + {(y[tr_idx] == 0).sum()} non-TB (all Chainda South)")

shortcut = new_model().fit(X[tr_idx], y[tr_idx])
s_score = lambda idx: shortcut.predict_proba(X[idx])[:, 1]
control = new_model().fit(X[kan_tr], y[kan_tr])               # control: an ordinary Kanyama model (both classes from Kanyama)
c_score = lambda idx: control.predict_proba(X[idx])[:, 1]

own_mix = np.r_[kan_te[y[kan_te] == 1], chai_te]              # held out, same recipe as the training set
pd.DataFrame({"shortcut model, its own mix (Kanyama TB + Chainda non-TB)": tb.screening_metrics(y[own_mix], s_score(own_mix), 0.5),
              "shortcut model, Kanyama alone": tb.screening_metrics(y[kan_te], s_score(kan_te), 0.5),
              "control model (ordinary Kanyama training), Kanyama alone": tb.screening_metrics(y[kan_te], c_score(kan_te), 0.5)}).T

training set: 94 TB cases (all Kanyama) + 149 non-TB (all Chainda South)


,people,TB cases,roc_auc,sensitivity,specificity,flag_rate
"shortcut model, its own mix (Kanyama TB + Chainda non-TB)",104.0,40.0,0.986,0.975,0.984,0.385
"shortcut model, Kanyama alone",259.0,40.0,0.472,0.975,0.009,0.988
"control model (ordinary Kanyama training), Kanyama alone",259.0,40.0,0.784,0.575,0.817,0.243


In [15]:
# What did it learn? Evidence 1: the images alone tell the two clinics apart (almost) perfectly.
both = np.r_[KAN, CHAI]
site = np.r_[np.zeros(len(KAN)), np.ones(len(CHAI))]
site_auc = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)), X[both], site, cv=5, scoring="roc_auc").mean()
print(f"site classifier (Kanyama vs Chainda South, from image features alone): ROC-AUC {site_auc:.3f}")

# Evidence 2: the shortcut model's "TB score" separates the clinics among people WITHOUT TB, but not TB from non-TB inside Kanyama
kan_neg_te, kan_tb_te = kan_te[y[kan_te] == 0], kan_te[y[kan_te] == 1]
idx = np.r_[kan_neg_te, chai_te]
auc_site = roc_auc_score(np.r_[np.zeros(len(kan_neg_te)), np.ones(len(chai_te))], -s_score(idx))
auc_disease = roc_auc_score(y[kan_te], s_score(kan_te))
print(f"shortcut score, Kanyama vs Chainda South among non-TB people: ROC-AUC {auc_site:.3f}  (high = it reads the clinic)")
print(f"shortcut score, TB vs non-TB inside Kanyama:                  ROC-AUC {auc_disease:.3f}  (0.5 = no disease signal)")

# Evidence 3: Chainda South's own 11 TB cases (none used in training) score like Chainda's healthy people
chai_tb = CHAI[y[CHAI] == 1]
groups5 = {"Kanyama, TB (held out)": kan_tb_te, "Kanyama, no TB (held out)": kan_neg_te,
           "Chainda South, TB (all 11)": chai_tb, "Chainda South, no TB (held out)": chai_te}
pd.DataFrame({"people": [len(g) for g in groups5.values()],
              "mean 'TB' score": [s_score(g).mean() for g in groups5.values()],
              "share flagged (score >= 0.5)": [(s_score(g) >= 0.5).mean() for g in groups5.values()]}, index=list(groups5))

site classifier (Kanyama vs Chainda South, from image features alone): ROC-AUC 0.991
shortcut score, Kanyama vs Chainda South among non-TB people: ROC-AUC 0.985  (high = it reads the clinic)
shortcut score, TB vs non-TB inside Kanyama:                  ROC-AUC 0.472  (0.5 = no disease signal)


,people,mean 'TB' score,share flagged (score >= 0.5)
"Kanyama, TB (held out)",40,0.949,0.975
"Kanyama, no TB (held out)",219,0.965,0.991
"Chainda South, TB (all 11)",11,0.019,0.000
"Chainda South, no TB (held out)",64,0.027,0.016


In [16]:
# Evidence 4: which direction in feature space did it learn? Compare its weights (cosine similarity, 1 = same direction)
# with a site classifier's weights and with an honest TB-vs-no-TB model's weights (Kanyama only). Common scaling for all three.
sc = StandardScaler().fit(X[both])
fit_w = lambda idx, lab: LogisticRegression(C=0.01, class_weight="balanced", max_iter=5000).fit(sc.transform(X[idx]), lab).coef_[0]
w_short = fit_w(tr_idx, y[tr_idx])
w_site = fit_w(np.r_[kan_tr, chai_tr], np.r_[np.zeros(len(kan_tr)), np.ones(len(chai_tr))])
w_tb = fit_w(kan_tr, y[kan_tr])
cos = lambda a, b: float(a @ b / np.linalg.norm(a) / np.linalg.norm(b))
pd.Series({"shortcut model vs site classifier": abs(cos(w_short, w_site)),
           "shortcut model vs honest TB model (Kanyama only)": abs(cos(w_short, w_tb)),
           "site classifier vs honest TB model": abs(cos(w_site, w_tb))}, name="cosine similarity (absolute value)").to_frame()

,cosine similarity (absolute value)
shortcut model vs site classifier,0.949
shortcut model vs honest TB model (Kanyama only),0.190
site classifier vs honest TB model,0.045


**Answer.** On its own mix it looks excellent; on Kanyama alone it is a coin toss. It learned the **machine**, not the disease.

- **Its own mix** (held-out Kanyama TB + Chainda non-TB): ROC-AUC 0.99, sensitivity 0.98, specificity 0.98.
- **Kanyama alone** (40 TB cases, 219 without TB): ROC-AUC 0.47, and it flags 99% of people (specificity 0.01). An ordinary Kanyama-trained model gets 0.78 on the same people.
- **Evidence:** (1) a classifier can tell Kanyama from Chainda South from the features alone: ROC-AUC 0.99. (2) The shortcut model's score separates the two clinics among people *without* TB (0.985) but separates TB from non-TB inside Kanyama at 0.47. (3) Chainda South's own 11 TB cases (never in training) get a mean score of 0.02, the same as Chainda's healthy people (0.03), and none is flagged. (4) Its weights point almost the same way as the site classifier's (cosine 0.95) and hardly the way of an honest TB model (0.19).

It is the Step 3 problem seen from the other side. A test set built the same confounded way (cases from one source, controls from another) gives a very good number and means nothing. In real projects this happens when cases come from a TB hospital and controls from a screening clinic, or when scanners differ by site. Defences: check that every site (and machine) has both cases and non-cases, test on a site-balanced set, and run the site-classifier check from the lesson. One split only, and 40 held-out TB cases: the size of the effect is not in doubt, the exact decimals are.

## Exercise 6

**A monitoring function.** Write a function that takes a batch (rows of `people`) and returns the header mismatch, feature drift and flag rate as a dict, ready for a Prefect task (compare lesson 06).

In [17]:
FEATURES_CHECKED_AGAINST = train       # the reference: step 1's training people


def monitor_batch(batch_idx, features=None):
    """Label-free monitoring of one batch of X-rays, before lab results exist.

    batch_idx: row positions in `people` (the batch). features: optional (n, 1024) array if the batch
    was processed differently (used below for the bug demo); default is X[batch_idx].
    Returns plain floats in a dict, so it can be a Prefect task's return value as it is.
    """
    batch_idx = np.asarray(batch_idx)
    F = X[batch_idx] if features is None else features
    batch = people.iloc[batch_idx]
    # 1. header mismatch: worst field, share of the batch with a rare/absent value
    mismatch = max(float((~batch[c].isin(NORMAL[c])).mean()) for c in HEADER_FIELDS)
    # 2. feature drift: can a classifier tell this batch from the reference? (ROC-AUC; 0.5 = no, 1.0 = perfectly)
    both = np.r_[X[FEATURES_CHECKED_AGAINST], F]
    which = np.r_[np.zeros(len(FEATURES_CHECKED_AGAINST)), np.ones(len(F))]
    drift = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)),
                            both, which, cv=5, scoring="roc_auc").mean()
    # 3. flag rate: share sent for a lab test at the production threshold
    flag_rate = float((model.predict_proba(F)[:, 1] >= threshold).mean())
    return {"n": int(len(batch_idx)), "header_mismatch": mismatch, "feature_drift": float(drift), "flag_rate": flag_rate}


steps = {"1. Kanyama, HIV-negative (held out)": test, "2. Chawama, HIV-negative": STEP2,
         "3. Chainda South, HIV-negative": STEP3, "4. Kanyama, HIV-positive": STEP4}
monitor = pd.DataFrame({name: monitor_batch(idx) for name, idx in steps.items()}).T
monitor

,n,header_mismatch,feature_drift,flag_rate
"1. Kanyama, HIV-negative (held out)",155.0,0.026,0.472,0.542
"2. Chawama, HIV-negative",512.0,0.000,0.650,0.414
"3. Chainda South, HIV-negative",148.0,0.993,0.995,0.000
"4. Kanyama, HIV-positive",346.0,0.026,0.593,0.590


In [18]:
# Does it reproduce the lesson? Compare with the numbers saved in tb_screening.ipynb's last checks table.
saved = nbformat.read("tb_screening.ipynb", 4)
cell = next(c for c in saved.cells if c.cell_type == "code" and 'checks["4. Kanyama, HIV-positive"]' in c.source)
lesson = {}
for line in cell.outputs[0]["data"]["text/plain"].splitlines():
    m = re.match(r"^(.*?)\s{2,}([\d.]+)\s+([\d.]+)\s+([\d.]+)\s*$", line)
    if m:
        lesson[m.group(1)] = [float(v) for v in m.groups()[1:]]
lesson = pd.DataFrame(lesson, index=["header_mismatch", "feature_drift", "flag_rate"]).T
ours = monitor[lesson.columns].astype(float)
print("largest difference to the lesson's printed values:", round(float((ours - lesson).abs().max().max()), 4))
assert np.allclose(ours, lesson, atol=0.0006), "monitor_batch disagrees with tb_screening.ipynb"
pd.concat({"monitor_batch": ours, "lesson notebook": lesson}, axis=1)

largest difference to the lesson's printed values: 0.0004


monitor_batch                         lesson notebook                        
                                    header_mismatch feature_drift flag_rate header_mismatch feature_drift flag_rate
1. Kanyama, HIV-negative (held out)           0.026         0.472     0.542           0.026         0.472     0.542
2. Chawama, HIV-negative                      0.000         0.650     0.414           0.000         0.650     0.414
3. Chainda South, HIV-negative                0.993         0.995     0.000           0.993         0.995     0.000
4. Kanyama, HIV-positive                      0.026         0.593     0.590           0.026         0.593     0.590

In [19]:
# The quiet bug (step 3b): headers still say "different machine", images are inverted by a pipeline that ignores them.
# Pass the wrongly processed features; the header check still fires, the flag rate looks plausible.
konica = [i for i in STEP3 if people.PhotometricInterpretation.iloc[i] == "MONOCHROME2"]
bug = tb.load_bug_features(people.anon_id.iloc[konica])
print(pd.Series(monitor_batch(konica, features=bug)).round(3).to_string())
print()
# What a Prefect task around it looks like (compare lesson 06); the dict is JSON-serialisable as it is:
print(json.dumps(monitor_batch(STEP2)))

n                  147.000
header_mismatch      1.000
feature_drift        1.000
flag_rate            0.361



{"n": 512, "header_mismatch": 0.0, "feature_drift": 0.649642796708759, "flag_rate": 0.4140625}


**Answer.** `monitor_batch(batch_idx)` returns `{"n", "header_mismatch", "feature_drift", "flag_rate"}` as plain numbers, so it can be a Prefect task's return value as it is. On the four steps it matches the lesson's table to its printed 3 decimals (largest difference 0.0004; the notebook asserts this). It also shows the quiet pipeline bug (step 3b): headers 100% different, drift 1.0, and a flag rate of 0.36 that looks normal, so the header check is the one that catches it.

As a Prefect flow (not run here; compare lesson 06):

```python
from prefect import flow, task

@task
def check_batch(batch_idx):
    return monitor_batch(batch_idx)

@flow
def screen_batch(batch_idx):
    rule = header_rule(batch_idx)              # Exercise 2: decide before scoring
    if rule["action"] == "block":
        raise RuntimeError(rule["message"])    # or send the message to rule["notify"]
    return check_batch(batch_idx)
```

Notes: the reference (training people, threshold, model) comes from this notebook's variables; in production load them from the registry. The function needs a batch of at least a few dozen images (the drift check is cross-validated). It cannot see population shift: step 4 stays quiet (drift 0.59, flag rate 0.59). Only lab results show that.